# Tutorial 1: Training a model

ElemeNet greatly simplifies the process of training deep learning models for chemistry. Models may be trained with one call to `training_pipeline`, driven either from Python (ideal for scripting) or from the `elemenet_train` CLI with a YAML config (ideal for SLURM). Both paths run the identical pipeline.

**Prerequisites:** install ElemeNet (see the top-level README) and run this notebook with the `ElemeNet` conda environment as the kernel.

## Option A: Python API

Every key below maps one-to-one to a `training_pipeline` keyword argument. Additional hyerparameters may be optionally specified in `model_config`, and will be preserved during subsequent hyperparameter optimization. Hyperparameter search may be disabled via `max_trials=None`, in which case any parameters specified in the `model_config` are used. For any unspecified hyperparameters, system defaults are used.

**Note:** Training takes considerable time and requires your own compute resources and dataset, so the cell below is meant to be edited and run on a GPU node, not executed inline while reading!

In [ ]:
from elemenet.training import training_pipeline


# Model architecture + objective.
model_config = {
    "encoder_type": "gnn",          # encoder architecture, may be "gnn" (2D) or "egnn" (equivariant 3D)
    "readout_type": "transformer",  # readout architecture, may be "mlp" or "transformer"
    "scope": "graph",               # "graph" for graph-level prediction
    "loss_type": "mse",             # mean squared error loss function
    "optimizer_type": "adamw",      # AdamW optimizer
}

preprocessing_kwargs = {
    "raw_data_path": [              # path to your dataset. for predefined splits, provide a list of paths: ["train.csv", "val.csv", "test.csv"]
        "datasets/QM9-train.csv",
        "datasets/QM9-val.csv",
        "datasets/QM9-test.csv",
    ],
    "train_val_test_split": None,   # training, validation, and test split ratios. use 'None' for predefined splits
}

training_pipeline(
    target_column="u0_atom",        # target property column(s). Add more for multitask: ["target_1", "target_2"]
    task="regression",              # task type (regression or classification)
    model_config=model_config,      # model architecture and training configuration
    graph_format="smiles",          # format of molecular graph (e.g., xyz, SMILES, mol2, etc.)
    mol_column="smiles",            # column in the dataset containing molecular graphs
    label_column="mol_id",          # column in the dataset containing molecule identifiers (optional)
    implicit_Hs=True,               # whether to use implicit hydrogens; default is False
    rdkit_features=True,            # whether to include additional RDKit features (defined for SMILES only); default is False
    max_trials=25,                  # number of Optuna trials. Set to 'None' to skip hyperparameter optimization and use system defaults
    epochs_hypersearch=100,         # epochs per hyperparameter search trial
    epochs=500,                     # epochs for training the final model
    early_stopping=True,            # whether to use checkpoint-based early stopping according to validation performance
    batch_size=64,                  # training batch size
    random_seed=42,                 # random seed for reproducibility
    save_dir="model",               # directory to save model checkpoints and logs
    preprocessing_kwargs=preprocessing_kwargs, # preprocessing configuration for data loading and processing
    device="cuda",                  # device to use for training (e.g., "cuda" or "cpu")
    save_embeddings=False,          # whether to save learned graph embeddings after training; default is True
)

Upon successful model training, your outputs will be saved in a run directory `model/` containing:

```
model/
├── config.yaml          # the exact parameters used (reusable as a CLI --config)
├── checkpoints/
│   ├── best.pt          # best-validation checkpoint
│   └── last.pt          # most recent checkpoint (use this to resume)
└── ...                  # data caches, embeddings, metrics
```

Because `config.yaml` is saved after training, you can reproduce or extend any run by pointing the CLI to its directory.

## Option B: CLI

The CLI takes the same parameters as a YAML file. The next cell writes a `config.yaml` next to this notebook.

In [ ]:
config_yaml = '''
target_column: u0_atom
task: regression
graph_format: smiles
mol_column: smiles
label_column: mol_id
implicit_Hs: true
rdkit_features: true

model_config:
  encoder_type: gnn
  readout_type: transformer
  loss_type: mse
  optimizer_type: adamw
  scope: graph

max_trials: 25
epochs_hypersearch: 100
epochs: 500
early_stopping: true
batch_size: 64
random_seed: 42
save_dir: model
device: cuda

preprocessing_kwargs:
  raw_data_path:
    - datasets/QM9-train.csv
    - datasets/QM9-val.csv
    - datasets/QM9-test.csv
  train_val_test_split: null
'''

with open("config.yaml", "w") as f:
    f.write(config_yaml)
print("wrote config.yaml")

Then train (the `!` runs it as a shell command from this notebook):

In [ ]:
!elemenet_train --config config.yaml

### On a SLURM cluster

For long runs, submit through the scheduler instead of running inline:

```bash
#!/bin/bash
#SBATCH --job-name=elemenet_train
#SBATCH -o training_test.out
#SBATCH --gres=gpu:1
#SBATCH --mem=64G
#SBATCH -c 4

conda activate ElemeNet

elemenet_train --config config.yaml
# alternatively: python training_test.py
```

## Hyperparameter search

Hyperparameter optimization may be performed with the following keywords:

- `max_trials` — number of search trials (set `max_trials=None` to skip the search and train a single model with defaults).
- `epochs_hypersearch` — epochs per trial (kept short relative to `epochs`).
- `quicksearch=true` — a faster, smaller search space.

The best hyperparameters are saved to `opt_params.yaml` and written into the run directory's `config.yaml`, so the final model and any later resume/fine-tune use them automatically.

## Switching to 3D (EGNN)

For geometry-aware models, set `encoder_type: egnn` and provide 3D structures:

```yaml
model_config:
  encoder_type: egnn
  readout_type: mlp
  loss_type: nll
  optimizer_type: adamw
  scope: graph
graph_format: xyz
mol_column: xyz
```

EGNN models build radius graphs via `torch_cluster`; that extension is installed by `install/install.sh`.